# v08 · Layer 3: word records filed under their own address
`PtolemyDesktop/Callimachus/hyperwebster_layer3.py` (git 2026-04-24, commit `3a5d7e6`). One JSON file per word, named by the v02 hex address, in a shard directory by first letter.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
l3 = load("PtolemyDesktop/Callimachus/hyperwebster_layer3.py", "l3")
import tempfile, pathlib
root = pathlib.Path(tempfile.mkdtemp())
B = l3.HyperWebsterLayer3(words_root=str(root))
n = B.ingest([("sedenion", 1000.0, "the sedenion is the I/O gate"),
              ("noether",  1001.0, "Noether conservation verified"),
              ("sedenion", 2000.0, "sedenion near zero divisor")])
print("records written/updated:", n)
p = B._record_path("sedenion")
print("file:", p.relative_to(root))
print("address_of('sedenion'):", B.address_of("sedenion"))
assert p.name == B.address_of("sedenion") + ".json"

records written/updated: 3
file: words_s/0000000000000000000000000000000000000000000000000008603286d52cc3.json
address_of('sedenion'): 0000000000000000000000000000000000000000000000000008603286d52cc3


## The Rabies Principle at the Python level: `first_encountered` is written once

In [3]:
r = B.lookup("sedenion")
print("hit_count:", r.hit_count, "| first_encountered:", r.first_encountered, "| timestamps:", r.timestamps, "| snippets:", len(r.snippets))
assert r.first_encountered == 1000.0 and r.hit_count == 2

hit_count: 2 | first_encountered: 1000.0 | timestamps: [1000.0, 1000.0, 2000.0] | snippets: 2


## Lookup is a computation, not a search
The path of a word is derived from the word; there is no index to consult. Two words never share a path (the address is a bijection), so no collision handling exists or is needed, **for words inside the charset**.

In [4]:
import itertools
words = ["a", "aa", "b", "sedenion", "Sedenion", "sedenion ", "0", "00"]
paths = [B._record_path(w).name for w in words]
print(len(set(paths)), "distinct filenames for", len(words), "words")
assert len(set(paths)) == len(words)

8 distinct filenames for 8 words
